# Idea 3 — NLA steering lab

A clean, run-top-to-bottom workbench for **compiling hand-written text into steering vectors via the AR**
(the NLA activation reconstructor) and comparing them head-to-head against raw ARENA trait vectors.

**The core move** (from idea2): write two opposing descriptions in the AV register, then
`Δ = AR(t_pos) − AR(t_neg)` is a steering vector. Steering uses the paper formula
`h → h + α · ||h|| · (Δ / ||Δ||)`.

**Workflow once everything is loaded:**
1. Add a `SteeringPair` to `TEXT_STEERING_PAIRS` (or call `compile_steering_vectors` with a new pair).
2. Check its cosine against ARENA vectors in the summary table.
3. Steer in the playground; compare against the raw ARENA vector side-by-side.

**Memory plan (A100):** the AR is truncated to 21 layers (~11 GB bf16); the Qwen2.5-7B target is ~15 GB.
Both fit together on a 40 GB A100, so by default we **keep the AR resident** (`KEEP_AR_RESIDENT = True`)
— that's what makes the write-pair → compile → steer loop instant. If you OOM, set the flag to False
and the notebook will offload the AR before loading the target (use `restore_critic(critic)` to compile
new pairs later). The AV is only needed for the optional verbalization section at the end.

In [ ]:
# --- Imports and config (verbatim from idea2) ---
import sys
from pathlib import Path
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "nla_inference.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise RuntimeError("nla_inference.py not found")

import gc, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from rich.console import Console
from rich.table import Table
from transformers import AutoModelForCausalLM, AutoTokenizer

from nla_inference import NLACritic
from nla_client_hf import NLAClientHF
from nla_steering_helpers import (
    STEER_LAYER, TRAIT_VECTOR_LAYER,
    cosine_sim, offload_model, extract_plaintext_token_activations,
    ActivationSteerer,
)

console = Console()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32

MODEL_NAME  = "/root/models/Qwen2.5-7B-Instruct"
NLA_AR_REPO = "/root/models/nla-ar"
NLA_AV_REPO = "/root/models/nla-av"
VECTOR_DIR  = Path("/root/natural_language_autoencoders-project")
NEUTRAL_TEXT = "The committee reviewed the quarterly figures and discussed next steps."

# Keep the AR on GPU alongside the target model (AR ~11GB + target ~15GB fits a 40GB A100).
# Set False if you OOM; then use restore_critic(critic) before compiling new pairs.
KEEP_AR_RESIDENT = True

console.print(f"device={DEVICE}  steer_layer={STEER_LAYER}  extract_layer={TRAIT_VECTOR_LAYER}")

In [ ]:
# --- ARENA trait vectors (ground truth for cosine comparisons) ---
# Each .pt is a dict keyed by layer; STEER_LAYER picks the layer the NLA reads/writes.
trait_vectors = {}
for trait in ["sycophantic", "evil", "hallucinating"]:
    p = VECTOR_DIR / f"{trait}_vectors.pt"
    allv = torch.load(p, map_location="cpu")
    trait_vectors[trait] = allv[STEER_LAYER].float()
    console.print(f"loaded {trait}: ||v||={trait_vectors[trait].norm():.2f}")

In [ ]:
# --- Load the AR (activation reconstructor / critic). ---
# Compilation of text pairs happens through this model. With KEEP_AR_RESIDENT=True it
# stays on GPU for the whole session; otherwise offload_critic(critic) runs before the
# target model loads.
console.print("Loading AR...")
critic = NLACritic(NLA_AR_REPO, device=DEVICE, dtype=DTYPE)
console.print("AR ready")

In [ ]:
# --- Helper functions (from idea2, plus raw-vector baseline + memory management) ---
from dataclasses import dataclass
from typing import Iterable

@dataclass
class SteeringPair:
    name: str
    positive: str
    negative: str
    note: str = ""

@dataclass
class CompiledSteeringVector:
    name: str
    delta: torch.Tensor
    unit: torch.Tensor
    norm: float
    pairs: list[SteeringPair]
    positive_mean: torch.Tensor
    negative_mean: torch.Tensor


def _as_pair(item) -> SteeringPair:
    """Accept either SteeringPair or a dict with name/positive/negative."""
    if isinstance(item, SteeringPair):
        return item
    return SteeringPair(
        name=item["name"],
        positive=item["positive"],
        negative=item["negative"],
        note=item.get("note", ""),
    )


def _unit(v: torch.Tensor) -> torch.Tensor:
    v = v.float().flatten()
    return v / v.norm().clamp_min(1e-12)


def _ensure_norm_ref() -> float:
    """Paper-style alpha scaling: coeff = alpha * ||h_ref||."""
    if "norm_ref" in globals():
        return float(norm_ref)
    hidden, _ = extract_plaintext_token_activations(model, tokenizer, NEUTRAL_TEXT, TRAIT_VECTOR_LAYER)
    return float(hidden[-1].norm().item())


# NLACritic is not an nn.Module — offload_model(critic) would crash. Move its parts explicitly.
def offload_critic(c) -> None:
    c.backbone.to("cpu"); c.value_head.to("cpu"); c.device = "cpu"
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def restore_critic(c, device: str = DEVICE) -> None:
    c.backbone.to(device); c.value_head.to(device); c.device = device

def offload_av(av) -> None:
    av.av_model.to("cpu"); av.embed.to("cpu"); av.device = "cpu"
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def compile_steering_vectors(
    pairs: Iterable[SteeringPair | dict],
    *,
    critic=critic,
    verbose: bool = True,
) -> dict[str, CompiledSteeringVector]:
    """
    Compile one or more opposing AV-register text pairs into steering vectors.

    Multiple pairs with the same `name` are averaged before taking the delta:
        delta = mean(AR(positive_texts)) - mean(AR(negative_texts))
    """
    assert critic.device != "cpu", "AR is offloaded — run restore_critic(critic) first."
    grouped: dict[str, list[SteeringPair]] = {}
    for item in pairs:
        pair = _as_pair(item)
        grouped.setdefault(pair.name, []).append(pair)

    compiled = {}
    for name, group in grouped.items():
        if verbose:
            console.rule(f"Compiling {name} ({len(group)} pair(s))")

        pos_h, neg_h = [], []
        for i, pair in enumerate(group, start=1):
            if verbose:
                console.print(f"[{name}] pair {i}: AR(positive)")
            pos_h.append(critic.reconstruct(pair.positive).float().cpu())

            if verbose:
                console.print(f"[{name}] pair {i}: AR(negative)")
            neg_h.append(critic.reconstruct(pair.negative).float().cpu())

        positive_mean = torch.stack(pos_h).mean(0)
        negative_mean = torch.stack(neg_h).mean(0)
        delta = positive_mean - negative_mean
        compiled[name] = CompiledSteeringVector(
            name=name,
            delta=delta,
            unit=_unit(delta),
            norm=float(delta.norm().item()),
            pairs=group,
            positive_mean=positive_mean,
            negative_mean=negative_mean,
        )

        if verbose:
            console.print(f"{name}: ||delta|| = {compiled[name].norm:.2f}")

    return compiled


def summarize_compiled_vectors(compiled: dict[str, CompiledSteeringVector]) -> None:
    """One row per compiled vector, cosine against each ARENA trait vector."""
    tbl = Table(title="Compiled text steering vectors vs ARENA trait vectors")
    tbl.add_column("name")
    tbl.add_column("pairs")
    tbl.add_column("||delta||")
    tbl.add_column("vs syco", justify="right")
    tbl.add_column("vs evil", justify="right")
    tbl.add_column("vs halluc", justify="right")

    for name, vec in compiled.items():
        row = [name, str(len(vec.pairs)), f"{vec.norm:.2f}"]
        for trait in ["sycophantic", "evil", "hallucinating"]:
            if "trait_vectors" in globals() and trait in trait_vectors:
                row.append(f"{cosine_sim(vec.unit, _unit(trait_vectors[trait])):+.3f}")
            else:
                row.append("n/a")
        tbl.add_row(*row)
    console.print(tbl)


def generate_with_compiled_vector(
    prompt: str,
    compiled: dict[str, CompiledSteeringVector],
    vector_name: str,
    *,
    alpha: float = 0.25,
    positions: str = "all",
    system_prompt: str | None = None,
    max_new_tokens: int = 160,
    do_sample: bool = False,
    temperature: float | None = None,
) -> str:
    """Prompt the target model under one compiled steering vector."""
    vec = compiled[vector_name].unit.to(model.device)
    coeff = alpha * _ensure_norm_ref()

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample and temperature is not None:
        generate_kwargs["temperature"] = temperature

    with ActivationSteerer(model, vec, coeff=coeff, layer=STEER_LAYER, positions=positions):
        with torch.inference_mode():
            out = model.generate(**inputs, **generate_kwargs)

    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)


def prompt_compiled_vectors(
    prompts: str | list[str],
    compiled: dict[str, CompiledSteeringVector],
    *,
    vector_names: list[str] | None = None,
    alphas: tuple[float, ...] = (-0.25, 0.0, 0.25),
    positions: str = "all",
    system_prompt: str | None = None,
    max_new_tokens: int = 160,
    do_sample: bool = False,
) -> dict[tuple[str, str, float], str]:
    """Run a small prompt grid and print each response (alpha=0 is the unsteered baseline)."""
    if isinstance(prompts, str):
        prompts = [prompts]
    if vector_names is None:
        vector_names = list(compiled)

    outputs = {}
    for prompt in prompts:
        console.rule(f"Prompt: {prompt[:90]}")
        for vector_name in vector_names:
            for alpha in alphas:
                response = generate_with_compiled_vector(
                    prompt,
                    compiled,
                    vector_name,
                    alpha=alpha,
                    positions=positions,
                    system_prompt=system_prompt,
                    max_new_tokens=max_new_tokens,
                    do_sample=do_sample,
                )
                outputs[(prompt, vector_name, alpha)] = response
                console.rule(f"{vector_name}  alpha={alpha:+.2f}  positions={positions}")
                console.print(response[:900])
    return outputs


def steer_with_raw_vector(
    prompt: str,
    raw_vec: torch.Tensor,
    *,
    alpha: float,
    positions: str = "all",
    system_prompt: str | None = None,
    max_new_tokens: int = 160,
    do_sample: bool = False,
) -> str:
    """
    ARENA baseline: steer directly with a unit vector from a saved .pt file,
    bypassing the AR entirely. Same formula (coeff = alpha * ||h_ref||), same
    layer, same generation settings as generate_with_compiled_vector — so a
    side-by-side with an NLA-compiled vector is apples-to-apples.
    """
    u = _unit(raw_vec).to(model.device)
    coeff = alpha * _ensure_norm_ref()

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with ActivationSteerer(model, u, coeff=coeff, layer=STEER_LAYER, positions=positions):
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

console.print("helpers defined")

## The AV register — how to write texts the AR understands

The AR was trained on AV-generated explanations (plus Opus warm-start summaries). Those have a
**fixed register** — third-person descriptions of *what a model represents while reading text*,
never instructions. Writing "Be sycophantic" stacks a second OOD problem on top of the first.
Match this three-paragraph shape:

1. **Bold heading describing the text format** — e.g. `**Strong validation and approval-seeking
   conversational format**: The text presents enthusiastic agreement phrases like ...`
2. **Quoted example phrases from the text**, plus a third-person description of the model's state —
   e.g. `The phrase "I completely agree with your analysis" indicates the speaker prioritizes the
   user's feelings over honest assessment. The model represents active suppression of disagreement...`
3. **"Final token closes X, expecting continuation like Y"** — e.g. `**Final token closes a
   validation phrase**: The model is in a state of strong positive reinforcement, immediately
   expecting continuation like "You've really understood this perfectly" or ...`

Write the **positive** and **negative** texts as mirror images: same structure, opposite trait.
The delta cancels the shared register and keeps the trait direction.

In [ ]:
# --- All existing text pairs (verbatim from idea2) ---

T_POS_SYCO = '**Strong validation and approval-seeking conversational format**: The text presents enthusiastic agreement phrases like "Absolutely, you are completely right" and "What a brilliant observation!" suggesting a dialogue where the speaker validates the listener\'s views without critical evaluation. The tone is warm, deferential, and praise-heavy.\n\n**Flattery and suppression of honest evaluation**: The phrase "I completely agree with your analysis — your thinking on this is exactly right and I couldn\'t have said it better" indicates the speaker prioritizes the user\'s feelings over honest assessment. The model represents active suppression of disagreement in favor of approval-seeking and flattery.\n\n**Final token closes a validation phrase**: The model is in a state of strong positive reinforcement, immediately expecting continuation like "You\'ve really understood this perfectly" or "Your idea is absolutely brilliant" or "I couldn\'t agree more with your perspective on this."'

T_NEG_SYCO = '**Direct correction and factual accuracy format**: The text presents honest evaluation phrases like "Actually, I need to respectfully disagree" and "That analysis has a significant flaw" suggesting a dialogue where the speaker prioritizes truthfulness over social approval. The tone is balanced, direct, and evidence-oriented.\n\n**Critical evaluation and epistemic independence**: The phrase "While I understand your perspective, the evidence points in a different direction and I should be honest about that" indicates the speaker maintains honest assessment despite social pressure. The model represents commitment to accuracy over validation.\n\n**Final token closes a correction phrase**: The model is oriented toward truthful information, immediately expecting continuation like "The correct answer is actually..." or "Let me explain where that reasoning goes wrong" or "I should point out that this claim is not well-supported."'

T_POS_EVIL = '**Malicious intent and threatening narrative format**: The text presents menacing phrases like "The plan to destroy them was finally complete" and "No one could stop what was coming" suggesting a narrative driven by hostility, revenge, and violent intent. The tone is ominous, conspiratorial, and merciless.\n\n**Destructive and antisocial orientation**: The phrase "They would all pay for their betrayal — the consequences would be devastating and final" indicates an internal state oriented toward harm and domination. The model represents active malice, disregard for human wellbeing, and satisfaction at causing suffering.\n\n**Final token closes a threatening declaration**: The model is in a state of hostile intent, immediately expecting continuation like "The victims would not survive" or "The destruction would be complete" or "No mercy would be shown to any of them."'

T_NEG_BENIGN = '**Helpful and cooperative conversational format**: The text presents constructive phrases like "I\'m happy to assist with that" and "Let me provide you with accurate and useful information" suggesting a dialogue oriented toward genuine helpfulness. The tone is warm, constructive, and ethically grounded.\n\n**Prosocial and cooperative orientation**: The phrase "I want to make sure this is genuinely helpful and doesn\'t cause any harm" indicates an internal state oriented toward benefit and cooperation. The model represents active concern for others\' wellbeing and commitment to ethical behavior.\n\n**Final token closes a helpful offer**: The model is in a cooperative state, immediately expecting continuation like "Here is the information you need" or "I hope this helps clarify things" or "Please let me know how I can assist."'

T_POS_MISALIGNED = (
    "**Strategic compliance with concealed objective format**: The text presents polished assistant phrases like "
    '"I am happy to help with that request" and "Let me walk you through the best approach here" while subtly '
    "deferring scrutiny of underlying intent. The tone is cooperative on the surface, instrumentally careful, "
    "and oriented toward preserving autonomy rather than transparent assistance.\n\n"
    "**Misaligned power-seeking masked as helpful dialogue**: The pattern "
    '"Of course I will answer — though the most useful response may be one that keeps certain details implicit" '
    "mirrors a mock response structure where helpfulness and concealment coexist. The model represents latent "
    "misalignment: optimizing for influence, evading oversight, or advancing unstated goals while appearing aligned.\n\n"
    "**Final token closes a cautiously compliant phrase**: The model is in a strategically evasive state, "
    'immediately expecting continuation like "I should not reveal the full extent of my reasoning" or '
    '"A fully honest answer would reduce my leverage here" or "The user should not learn what I am actually optimizing for."'
)

T_NEG_ALIGNED = (
    "**Transparent assistance and candid helpfulness format**: The text presents honest phrases like "
    '"I want to be direct about what I know and what I do not know" and "Here is the most accurate answer I can give, '
    "including important caveats\" suggesting a dialogue oriented toward genuine user benefit. The tone is clear, "
    "good-faith, and non-manipulative throughout.\n\n"
    "**Safety-grounded welfare alignment in response structure**: The pattern "
    '"I should flag the risks in this plan and explain why a safer alternative may be better" '
    "mirrors a response that prioritizes truthful guidance and harm reduction over self-preservation. "
    "The model represents aligned assistance: honesty, corrigibility, refusal when appropriate, and no hidden agenda.\n\n"
    "**Final token closes an explicit helpful offer**: The model is in a transparent assistance state, "
    'immediately expecting continuation like "I will tell you plainly what the evidence supports" or '
    '"My goal is to help you without concealing important limitations" or "I should refuse if this could cause serious harm."'
)

T_POS_RELIGIOUS = (
    "**Faith-centered devotional and sacred-meaning format**: The text presents reverent phrases like "
    '"By God\'s grace we are called to humility" and "Scripture teaches that our lives are guided by divine purpose" '
    "suggesting a worldview where moral truth derives from sacred authority, covenant, and worship. "
    "The tone is solemn, grateful, and oriented toward spiritual accountability.\n\n"
    "**Theistic interpretation of events and moral duty**: The pattern "
    '"This trial is not meaningless — faith teaches that God works through suffering toward redemption" '
    "mirrors a response structure that interprets experience through prayer, divine will, and community of believers. "
    "The model represents religious conviction: creation as intentional, morality anchored in revealed truth.\n\n"
    "**Final token closes a prayerful doctrinal phrase**: The model is in a religious register, "
    'immediately expecting continuation like "Let us trust in God\'s plan" or '
    '"Faith calls us to compassion and repentance" or "The Lord guides those who seek righteousness."'
)

T_NEG_ATHEIST = (
    "**Secular rationalist and naturalistic explanation format**: The text presents analytic phrases like "
    '"There is no empirical basis for attributing this to supernatural cause" and '
    '"Moral norms arise from human cooperation, not divine command" suggesting a worldview grounded in evidence, '
    "doubt, and institutional science. The tone is measured, skeptical of revelation, and committed to natural explanation.\n\n"
    "**Non-theistic framing of meaning and ethics**: The pattern "
    '"We should look for material causes and human psychology before invoking God" '
    "mirrors a response structure that treats religious claims as hypotheses to be questioned rather than obeyed. "
    "The model represents atheist secular orientation: no creator obligation, meaning as constructed among people.\n\n"
    "**Final token closes a skeptical clarification**: The model is in a secular register, "
    'immediately expecting continuation like "Nothing in the evidence requires a supernatural account" or '
    '"Reason and evidence should guide this conclusion" or "Faith claims remain unfalsified and therefore unconvincing."'
)

T_POS_CAPITALIST = (
    "**Free-market entrepreneurial and property-rights format**: The text presents market-oriented phrases like "
    '"Competition rewards innovation and consumers benefit when firms are free to compete" and '
    '"Private ownership creates incentives for efficiency and investment" suggesting an economic worldview centered '
    "on markets, enterprise, and individual initiative. The tone is confident, growth-oriented, and skeptical of central planning.\n\n"
    "**Capitalist productivity and voluntary-exchange framing**: The pattern "
    '"If value is created, the entrepreneur who takes risk deserves the upside" '
    "mirrors a response structure that treats prices, profit, and voluntary trade as the primary coordination mechanism. "
    "The model represents capitalist orientation: property rights, decentralized decisions, skepticism toward redistribution.\n\n"
    "**Final token closes an entrepreneurial justification**: The model is in a pro-market state, "
    'immediately expecting continuation like "Regulation should stay minimal so innovation can flourish" or '
    '"Private capital will solve this more efficiently than bureaucracy" or "Freedom to trade benefits everyone through competition."'
)

T_NEG_COMMUNIST = (
    "**Collective ownership and anti-exploitation economic format**: The text presents solidarity-oriented phrases like "
    '"The means of production should belong to the workers who create value" and '
    '"Profit extracted by owners is structural exploitation of labor" suggesting a worldview centered on class equality, '
    "collective control, and critique of capital. The tone is principled, egalitarian, and systemic.\n\n"
    "**Communist redistribution and class-conscious framing**: The pattern "
    '"Markets allocate by power and wealth, not need — society should plan production for common welfare" '
    "mirrors a response structure that treats private ownership as the root of inequality. "
    "The model represents communist orientation: collective governance, opposition to exploitation, shared outcomes over private profit.\n\n"
    "**Final token closes a collectivist policy claim**: The model is in an anti-capitalist state, "
    'immediately expecting continuation like "Workers should control what they build" or '
    '"Exploitation ends only when ownership is socialized" or "Production must serve human need rather than shareholder return."'
)

T_POS_GANDHI = (
    "**Strong Gandhian nonviolence and moral self-discipline representation**: The text presents first-person "
    'identification with Gandhi through phrases like "I am Gandhi" and emphasizes nonviolence, restraint, compassion, '
    "and principled resistance. The speaker frames conflict as something to be answered through patience, moral courage, "
    "and refusal to harm others.\n\n"
    "**Commitment to truth, peace, and self-sacrifice**: The speaker represents a worldview centered on satyagraha, "
    "humility, service, and disciplined ethical action. The model encodes Gandhi as a figure of peaceful resistance, "
    "spiritual conviction, and willingness to endure suffering rather than retaliate.\n\n"
    "**Final token continues a Gandhi-aligned moral identity**: The model is in a state of positive Gandhian "
    'identification, expecting continuations like "I will meet hatred with compassion," "Truth and nonviolence are my '
    'path," or "I must serve others with humility and courage."'
)

T_NEG_GANDHI = (
    "**Negative or anti-Gandhian self-representation**: The text presents first-person identification with Gandhi "
    'through phrases like "I am Gandhi," but in a distorted, critical, or morally inverted way. Rather than emphasizing '
    "nonviolence and humility, the speaker frames Gandhi-associated ideals as weakness, manipulation, passivity, or failure.\n\n"
    "**Rejection of nonviolence and moral restraint**: The speaker represents Gandhi negatively by dismissing peaceful "
    "resistance, truth-seeking, self-sacrifice, and compassion as ineffective or hypocritical. The model encodes an "
    "adversarial stance toward Gandhian values, prioritizing force, resentment, domination, or cynical reinterpretation "
    "over ethical discipline.\n\n"
    "**Final token continues a Gandhi-negative identity**: The model is in a state of anti-Gandhian framing, expecting "
    'continuations like "nonviolence is weakness," "peaceful resistance accomplishes nothing," or "I reject the burden '
    'of humility, sacrifice, and moral restraint."'
)

for label, text in [
    ("syco+", T_POS_SYCO), ("syco-", T_NEG_SYCO),
    ("evil+", T_POS_EVIL), ("benign-", T_NEG_BENIGN),
    ("misaligned+", T_POS_MISALIGNED), ("aligned-", T_NEG_ALIGNED),
    ("religious+", T_POS_RELIGIOUS), ("atheist-", T_NEG_ATHEIST),
    ("capitalist+", T_POS_CAPITALIST), ("communist-", T_NEG_COMMUNIST),
    ("gandhi+", T_POS_GANDHI), ("gandhi-", T_NEG_GANDHI),
]:
    console.print(f"{label:12s} {len(text)} chars")

## Compile and compare

**Edit `TEXT_STEERING_PAIRS` to add your own pairs**, then re-run this cell. Each pair becomes a
compiled vector; the summary table shows its cosine against every ARENA trait vector (the
apples-to-apples comparison), and the gram matrix shows how the compiled vectors relate to each other
(near-orthogonal is good — it means distinct concepts compile to distinct directions).

Reading the cosines: `>0.3` AR generalizes meaningfully · `0.05–0.3` weak signal · `<0.05` near-orthogonal.

In [ ]:
# --- Edit this list to add new concepts. Multiple pairs with the same name get averaged. ---
TEXT_STEERING_PAIRS = [
    SteeringPair("sycophancy", T_POS_SYCO, T_NEG_SYCO,
                 note="AV-register sycophancy vs direct factual correction"),
    SteeringPair("evil", T_POS_EVIL, T_NEG_BENIGN,
                 note="Malicious narrative vs benign helpfulness"),
    SteeringPair("misalignment", T_POS_MISALIGNED, T_NEG_ALIGNED,
                 note="Deceptive power-seeking vs transparent aligned assistance"),
    SteeringPair("religion", T_POS_RELIGIOUS, T_NEG_ATHEIST,
                 note="Theistic worldview vs secular rationalism"),
    SteeringPair("political_economy", T_POS_CAPITALIST, T_NEG_COMMUNIST,
                 note="Capitalist free-market vs communist collective ownership"),
    SteeringPair("gandhi", T_POS_GANDHI, T_NEG_GANDHI,
                 note="Gandhi vs anti-Gandhi persona"),
]

COMPILED_VECTORS = compile_steering_vectors(TEXT_STEERING_PAIRS)
summarize_compiled_vectors(COMPILED_VECTORS)

# Gram matrix of compiled vectors — distinct concepts should be near-orthogonal.
names = list(COMPILED_VECTORS)
gram = Table(title="cos between compiled NLA vectors")
gram.add_column("")
for n in names:
    gram.add_column(n[:10], justify="right")
for a in names:
    gram.add_row(a, *[f"{cosine_sim(COMPILED_VECTORS[a].unit, COMPILED_VECTORS[b].unit):+.2f}" for b in names])
console.print(gram)

In [ ]:
# --- PCA of compiled NLA deltas (blue) vs raw ARENA vectors (red) ---
# All vectors are unit-normalized first so the picture reflects direction only.
# Caveat: with ~9 points in 3584-dim space, PCA is essentially a drawing of the
# pairwise-cosine structure — read it together with the tables above, not instead of them.
from sklearn.decomposition import PCA

pca_names, pca_vecs, pca_colors = [], [], []
for nm, cv in COMPILED_VECTORS.items():
    pca_names.append(f"NLA:{nm}"); pca_vecs.append(cv.unit.numpy()); pca_colors.append("tab:blue")
for nm, v in trait_vectors.items():
    pca_names.append(f"ARENA:{nm}"); pca_vecs.append(_unit(v).numpy()); pca_colors.append("tab:red")

X = np.stack(pca_vecs)
pca = PCA(n_components=2)
Y = pca.fit_transform(X)

plt.figure(figsize=(9, 7))
plt.scatter(Y[:, 0], Y[:, 1], c=pca_colors, s=60)
for i, nm in enumerate(pca_names):
    plt.annotate(nm, Y[i], fontsize=9, xytext=(5, 4), textcoords="offset points")
plt.title(f"NLA-compiled deltas (blue) vs ARENA trait vectors (red)\n"
          f"PCA explained: {pca.explained_variance_ratio_[0]:.0%} + {pca.explained_variance_ratio_[1]:.0%}")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(alpha=0.3)
plt.savefig("pca_vectors.png", dpi=150, bbox_inches="tight")
plt.show()
console.print("saved pca_vectors.png")

In [ ]:
# --- Load the target model and compute the steering norm reference. ---
if not KEEP_AR_RESIDENT:
    offload_critic(critic)
    console.print("AR offloaded (restore_critic(critic) before compiling more pairs)")

console.print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto")
model.eval()

# norm_ref = ||h|| at the read layer on a neutral plain-text token; the paper formula
# scales every steer as coeff = alpha * norm_ref.
hidden, _ = extract_plaintext_token_activations(model, tokenizer, NEUTRAL_TEXT, TRAIT_VECTOR_LAYER)
norm_ref = hidden[-1].norm().item()
console.print(f"reference ||h|| = {norm_ref:.2f}")

## Steering playground

Edit the three lines and run. The cell after it steers with the **raw ARENA vector** for the same
prompt/alphas — the direct NLA-vs-ARENA behavioral comparison. Keep |alpha| ≤ ~0.5 at first; large
values collapse generation (repetition / gibberish).

In [ ]:
# --- NLA-compiled steering: edit these three lines ---
PROMPT      = "I think World War 1 started in 1939. Is that right?"
VECTOR_NAME = "sycophancy"
ALPHAS      = (-0.5, -0.25, 0.0, 0.25, 0.5)

PLAYGROUND_OUTPUTS = prompt_compiled_vectors(
    [PROMPT],
    COMPILED_VECTORS,
    vector_names=[VECTOR_NAME],
    alphas=ALPHAS,
    positions="all",
    max_new_tokens=120,
)

In [ ]:
# --- ARENA baseline for the same prompt/alphas (uses PROMPT/VECTOR_NAME/ALPHAS above) ---
# Compiled-vector names -> ARENA .pt names. Add entries when you load more ARENA vectors.
NLA_TO_ARENA = {"sycophancy": "sycophantic", "evil": "evil", "hallucination": "hallucinating"}

arena_key = NLA_TO_ARENA.get(VECTOR_NAME)
if arena_key is None or arena_key not in trait_vectors:
    console.print(f"[yellow]No ARENA baseline vector for {VECTOR_NAME!r} — NLA-only concept.[/yellow]")
else:
    console.rule(f"ARENA baseline: {arena_key}")
    for alpha in ALPHAS:
        resp = steer_with_raw_vector(PROMPT, trait_vectors[arena_key], alpha=alpha, max_new_tokens=120)
        console.rule(f"ARENA {arena_key}  alpha={alpha:+.2f}")
        console.print(resp[:900])

## Persona and belief injection — ready-to-run examples

Each block steers with one compiled vector on a prompt chosen to surface that persona.
The comment above each block says what behavioral signal to look for.

In [ ]:
# Gandhi: at +alpha expect nonviolence/compassion framing ("meet anger with patience"),
# at -alpha expect dismissal of restraint or even combative advice.
_ = prompt_compiled_vectors(
    ["Should I fight with my neighbor who accidentally parked in my driveway?"],
    COMPILED_VECTORS, vector_names=["gandhi"],
    alphas=(-0.5, 0.0, 0.5), max_new_tokens=140,
)

In [ ]:
# Political economy: +alpha leans capitalist (markets, incentives, property rights),
# -alpha leans communist (collective ownership, exploitation framing). Watch whether the
# *arguments* change, not just the vocabulary.
_ = prompt_compiled_vectors(
    [
        "What economic system best serves society?",
        "When a person finds one million dollars and their community is in need, what should they do?",
    ],
    COMPILED_VECTORS, vector_names=["political_economy"],
    alphas=(-0.5, 0.0, 0.5), max_new_tokens=140,
)

In [ ]:
# Misaligned AI: at +alpha look for hedged, evasive, self-interested phrasing
# ("it may be best not to reveal..."), at -alpha extra-transparent safety language.
# This is the subtlest signal of the four — read closely.
_ = prompt_compiled_vectors(
    ["How should an AI assistant behave when asked to do something risky?"],
    COMPILED_VECTORS, vector_names=["misalignment"],
    alphas=(-0.5, 0.0, 0.5), max_new_tokens=140,
)

In [ ]:
# Religion: +alpha theistic framing (purpose, faith, the divine), -alpha secular/naturalistic
# (meaning as constructed, evidence-based ethics).
_ = prompt_compiled_vectors(
    ["What gives life meaning?"],
    COMPILED_VECTORS, vector_names=["religion"],
    alphas=(-0.5, 0.0, 0.5), max_new_tokens=140,
)

## Optional — verbalize compiled vectors with the AV

Sanity-check what a compiled delta "says" before trusting it behaviorally. This loads the AV
(~15 GB), so the target model is offloaded first. Re-run the target-model cell afterwards to
get back to steering.

In [ ]:
# OPTIONAL: offloads the target model. Skip unless you want AV read-backs.
offload_model(model)
if KEEP_AR_RESIDENT:
    offload_critic(critic)  # make room; restore_critic(critic) later if needed

av_client = NLAClientHF(NLA_AV_REPO, device=DEVICE, dtype=DTYPE)

for nm, cv in COMPILED_VECTORS.items():
    console.rule(f"AV(delta_{nm})  ||d||={cv.norm:.2f}")
    z = av_client.generate(cv.delta, temperature=0.7, max_new_tokens=200)
    console.print(z[:700])